# ML Feature Engineering — Base Features (Phase A) & Split Manifests (Phase B)

Olist Brazilian E-Commerce — IT5006 Group 11 · ML dataset pipeline.

This notebook **explains and calls** the reusable modules under `src/features/`
(per `data/business/ml/README.md` §1) and reproduces, end-to-end and
deterministically, every artifact under `data/business/ml/`:

| Artifact | Phase |
| --- | --- |
| `orders_ml_features.csv` | A |
| `feature_schema.json` | A |
| `dataset_manifest.json` | A |
| `zip_centroids.csv` | A |
| `quality_report.json` | A |
| `split_assignments.csv` | B |
| `cv_assignments.csv` | B |
| `split_report.json` | B |

**Lineage:** `data/preprocessed/*.csv → src/features/ → data/business/ml/`.
ML never reads `data/raw/` or `data/business/dashboard/`.


## 0. Environment & repo root

Locate the repository root relative to wherever the notebook runs, so the
`src.features` package resolves without hard-coded paths.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "preprocessed").is_dir() and (p / "src" / "features").is_dir():
            return p
    raise FileNotFoundError("repo root not found (needs data/preprocessed and src/features)")

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

import numpy, pandas, sklearn, geopandas, shapely, pyproj
print("pandas", pandas.__version__, "| numpy", numpy.__version__)
print("sklearn", sklearn.__version__, "| geopandas", geopandas.__version__,
      "| shapely", shapely.__version__, "| pyproj", pyproj.__version__)


## 1. Input inventory

The only transactional inputs are `data/preprocessed/*.csv`. The single
external reference is the pinned IBGE `BR_Pais_2024` country polygon under
`docs/references/geography/` (a documented, label-independent quality-control
exception). Row counts below mirror `dataset_manifest.json`.


In [ ]:
PRE = ROOT / "data" / "preprocessed"
ML = ROOT / "data" / "business" / "ml"

print("preprocessed inputs:")
for f in sorted(PRE.glob("*.csv")):
    with open(f, encoding="utf-8") as fh:
        n = sum(1 for _ in fh) - 1
    print(f"  {f.name:45s} {n:>8,} rows")

print()
print("boundary reference dir:", ROOT / "docs" / "references" / "geography")
print("(lineage: data/preprocessed -> src/features -> data/business/ml; "
       "no data/raw or dashboard read)")


## 2. Reusable implementation (`src/features`)

All derivation logic lives in the modules below; their full, verbatim source is
embedded here so the notebook is self-contained and reviewable. The driver cells
in sections 3–4 call these exact modules, so running this notebook reproduces
byte-identical data files.


### `src/features/__init__.py`

Package docstring and the two CLI entrypoints.

```python
"""Feature engineering and split manifest modules (feature_contract.md v1.1).

This package implements the deterministic ``data/preprocessed`` -> ``data/business/ml``
lineage described in ``data/business/ml/feature_contract.md`` and
``data/business/ml/README.md``. Entrypoints:

    python -m src.features.build_features   # base table + schema + manifests + quality report
    python -m src.features.create_splits    # outer holdout + inner CV manifests
"""

```


### `src/features/contract.py`

Frozen constants (27-column predictor allowlist, valid UF codes, usable payment types), column roles, and the serialized `feature_schema`.

```python
"""Constants and column contracts for the ML base table.

This mirrors ``data/business/ml/feature_contract.md`` (v1.1). The 27-column
predictor allowlist is frozen here; targets, identifiers, eligibility and audit
columns are declared with explicit roles so that training never infers predictors
by dropping a target or selecting "all numeric columns".
"""
from __future__ import annotations

import pandas as pd

# ---------------------------------------------------------------------------
# Frozen constants
# ---------------------------------------------------------------------------

# 27 Brazilian federative-unit codes (feature_contract.md §4 geography step 2).
VALID_STATES = frozenset({
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG",
    "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
})

# Usable payment types (feature_contract.md §4). Normalized lowercase + trimmed.
USABLE_PAYMENT_TYPES = frozenset({"credit_card", "boleto", "voucher", "debit_card"})

UNKNOWN = "Unknown"

# Version strings shared across artifacts (build + splits).
FEATURE_VERSION = "v1.1"
SPLIT_VERSION = "v1.0"


def normalize_state(value) -> str:
    """Normalize a Brazilian UF code; return ``UNKNOWN`` for anything invalid."""
    if value is None:
        return UNKNOWN
    if isinstance(value, float) and pd.isna(value):
        return UNKNOWN
    s = str(value).strip().upper()
    return s if s in VALID_STATES else UNKNOWN

# Exact 27-column predictor allowlist, in the contract's frozen order.
PREDICTOR_ALLOWLIST = [
    "n_items", "has_items", "n_products", "n_sellers", "n_categories",
    "total_price", "total_freight", "freight_ratio", "freight_ratio_missing",
    "total_weight_g", "total_volume_cm3", "weight_missing_fraction",
    "volume_missing_fraction", "primary_category", "category_missing_fraction",
    "customer_state", "primary_seller_state", "distance_km_max",
    "distance_missing_fraction", "interstate_share",
    "purchase_month", "purchase_dayofweek", "purchase_hour",
    "primary_payment_type", "payment_installments_max", "n_payment_methods",
    "payment_missing",
]

# Categorical predictors (downstream OHE with handle_unknown='ignore').
CATEGORICAL_COLUMNS = [
    "customer_state", "primary_seller_state", "primary_category",
    "primary_payment_type", "purchase_month", "purchase_dayofweek", "purchase_hour",
]

# Numeric predictors = allowlist minus categoricals.
NUMERIC_COLUMNS = [c for c in PREDICTOR_ALLOWLIST if c not in CATEGORICAL_COLUMNS]

# ---------------------------------------------------------------------------
# Base-table column roles (export order for orders_ml_features.csv)
# ---------------------------------------------------------------------------

IDENTIFIER_COLUMNS = ["order_id", "customer_id", "customer_unique_id"]
TARGET_COLUMNS = ["lead_days", "review_score_min", "is_detractor"]
ELIGIBILITY_COLUMNS = ["eligible_regression", "eligible_classification"]
AUDIT_COLUMNS = [
    "prediction_timestamp", "order_status",
    "regression_exclusion_reason", "classification_exclusion_reason",
    "regression_label_available_at", "review_latest_observed_at",
    "review_count", "seller_id_missing",
]

# Export order: identifiers, audit anchor, targets, eligibility, audit, features.
BASE_COLUMN_ORDER = (
    ["order_id", "customer_id", "customer_unique_id",
     "prediction_timestamp", "order_status",
     "lead_days", "review_score_min", "is_detractor",
     "eligible_regression", "eligible_classification",
     "regression_exclusion_reason", "classification_exclusion_reason",
     "regression_label_available_at", "review_latest_observed_at",
     "review_count", "seller_id_missing"]
    + PREDICTOR_ALLOWLIST
)

# ---------------------------------------------------------------------------
# Per-column schema (serialized to feature_schema.json)
# ---------------------------------------------------------------------------

def _role(role: str, **kw) -> dict:
    d = {"role": role}
    d.update(kw)
    return d


FEATURE_SCHEMA = [
    _role("identifier", name="order_id", dtype="string",
          description="Source order key; unique non-null; audit/join only."),
    _role("identifier", name="customer_id", dtype="string",
          description="Session key; audit/join only, never a predictor."),
    _role("identifier", name="customer_unique_id", dtype="string",
          description="Persistent customer key used for CV grouping."),
    _role("audit", name="prediction_timestamp", dtype="datetime",
          description="Checkout cutoff anchor (order_purchase_timestamp); splitting only."),
    _role("audit", name="order_status", dtype="string",
          description="Final source status; audit only, never a predictor."),
    _role("target", name="lead_days", dtype="float", unit="days",
          description="(delivered - purchase) seconds / 86400; delivered + strictly positive."),
    _role("target_source", name="review_score_min", dtype="int",
          description="Minimum valid score 1-5 across supplied reviews; null if none."),
    _role("target", name="is_detractor", dtype="int",
          description="1 if review_score_min <= 2; 0 if >= 3; null if no valid score."),
    _role("eligibility", name="eligible_regression", dtype="int",
          description="Delivered + valid positive lead_days + valid prediction time + resolved group."),
    _role("eligibility", name="eligible_classification", dtype="int",
          description="Valid review target + valid prediction time + resolved group; delivery not required."),
    _role("audit", name="regression_exclusion_reason", dtype="string",
          description="Ordered reason codes for regression ineligibility."),
    _role("audit", name="classification_exclusion_reason", dtype="string",
          description="Ordered reason codes for classification ineligibility."),
    _role("audit", name="regression_label_available_at", dtype="datetime",
          description="Customer delivery timestamp for valid regression outcomes."),
    _role("audit", name="review_latest_observed_at", dtype="datetime",
          description="Max valid answer timestamp across contributing valid reviews."),
    _role("audit", name="review_count", dtype="int",
          description="Number of review records for the order."),
    _role("audit", name="seller_id_missing", dtype="int",
          description="Any item seller_id missing -> complete seller set unknown."),

    _role("feature", name="n_items", dtype="int", phase="base",
          description="Count of item rows per order."),
    _role("feature", name="has_items", dtype="int", phase="base",
          description="int(n_items > 0)."),
    _role("feature", name="n_products", dtype="int", phase="base",
          description="Distinct non-null product_id among items."),
    _role("feature", name="n_sellers", dtype="int", phase="base",
          description="Distinct non-null seller_id among items."),
    _role("feature", name="n_categories", dtype="int", phase="base",
          description="Distinct resolved translated English categories, excluding Unknown."),
    _role("feature", name="total_price", dtype="float", unit="BRL", phase="base",
          description="Sum item price, each item once."),
    _role("feature", name="total_freight", dtype="float", unit="BRL", phase="base",
          description="Sum item freight_value, each item once."),
    _role("feature", name="freight_ratio", dtype="float", phase="base",
          description="total_freight / total_price; null if denominator <= 0 or either missing."),
    _role("feature", name="freight_ratio_missing", dtype="int", phase="base",
          description="int(freight_ratio is null)."),
    _role("feature", name="total_weight_g", dtype="float", unit="g", phase="base",
          description="Sum product_weight_g; null if any item weight invalid/missing or no items."),
    _role("feature", name="total_volume_cm3", dtype="float", unit="cm^3", phase="base",
          description="Sum length*height*width; null if any item volume unknown or no items."),
    _role("feature", name="weight_missing_fraction", dtype="float", phase="base",
          description="Invalid/unknown item weights / n_items; null if no items."),
    _role("feature", name="volume_missing_fraction", dtype="float", phase="base",
          description="Invalid/unknown item volumes / n_items; null if no items."),
    _role("feature", name="primary_category", dtype="string", phase="base",
          description="English category of the deterministic primary item."),
    _role("feature", name="category_missing_fraction", dtype="float", phase="base",
          description="Fraction of items lacking a resolved category; null if no items."),
    _role("feature", name="customer_state", dtype="string", phase="base",
          description="Customer state, normalized; invalid -> Unknown."),
    _role("feature", name="primary_seller_state", dtype="string", phase="base",
          description="Seller state of the primary item; Unknown for absent/unresolved."),
    _role("feature", name="distance_km_max", dtype="float", unit="km", phase="base",
          description="Max Haversine distance from customer to each distinct seller."),
    _role("feature", name="distance_missing_fraction", dtype="float", phase="base",
          description="Distinct sellers with unresolved distance / n_sellers; null if no sellers."),
    _role("feature", name="interstate_share", dtype="float", phase="base",
          description="Fraction of distinct sellers whose state differs from customer state."),
    _role("feature", name="purchase_month", dtype="int", phase="base",
          description="Purchase month 1-12, categorical."),
    _role("feature", name="purchase_dayofweek", dtype="int", phase="base",
          description="Purchase day of week 0-6 (Monday=0), categorical."),
    _role("feature", name="purchase_hour", dtype="int", phase="base",
          description="Purchase hour 0-23, categorical."),
    _role("feature", name="primary_payment_type", dtype="string", phase="base",
          description="Payment type with largest summed value; ties lexicographic; Unknown if none usable."),
    _role("feature", name="payment_installments_max", dtype="int", phase="base",
          description="Max positive recorded installments across usable payment rows."),
    _role("feature", name="n_payment_methods", dtype="int", phase="base",
          description="Distinct usable payment types."),
    _role("feature", name="payment_missing", dtype="int", phase="base",
          description="int(no usable payment rows)."),
]


def role_map() -> dict[str, str]:
    """column name -> role."""
    return {c["name"]: c["role"] for c in FEATURE_SCHEMA}

```


### `src/features/geography.py`

IBGE boundary preparation, ZIP-prefix normalization, Haversine distance, polygon filtering and the 8-step ZIP/state median lookup, plus per-order distance/interstate aggregation.

```python
"""Geography lookup: IBGE boundary reference + polygon-filtered ZIP/state medians.

Implements feature_contract.md v1.1 §4 "Geography lookup: required algorithm",
"Haversine and audit outputs" and the boundary provenance rules.

The only transactional inputs are ``data/preprocessed/*.csv``. The single external
reference is the pinned IBGE ``BR_Pais_2024`` country polygon staged under
``docs/references/geography/`` (a documented, label-independent quality-control
exception).
"""
from __future__ import annotations

import hashlib
import json
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely import contains_xy

from .contract import UNKNOWN, VALID_STATES, normalize_state

EARTH_RADIUS_KM = 6371.0

# Archive SHA-256 recorded at staging time (see boundary_manifest.json).
_ARCHIVE_SHA256 = "e84c4d4ab199e646b5a180e0f5b7a991fe4c3d424dff8ae3dcf4495992c128d5"


def geography_dir(repo_root: Path) -> Path:
    return Path(repo_root) / "docs" / "references" / "geography"


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def normalize_zip(value) -> str | None:
    """Zero-pad an integer/string ZIP prefix to a 5-digit string; reject others."""
    if value is None:
        return None
    if isinstance(value, float):
        if not np.isfinite(value) or not value.is_integer():
            return None
        value = int(value)
    if isinstance(value, (int, np.integer)):
        ival = int(value)
    elif isinstance(value, str):
        s = value.strip()
        if not s.isdigit():
            return None
        ival = int(s)
    else:
        return None
    if ival < 0 or ival > 99999:
        return None
    return f"{ival:05d}"


def haversine_km(lat1, lon1, lat2, lon2) -> np.ndarray:
    """Straight-line distance (km) on a 6371 km sphere.

    feature_contract.md: a = sin^2(dlat/2) + cos(lat1)cos(lat2)sin^2(dlon/2),
    clip a to [0,1], distance = 2*6371*arcsin(sqrt(a)).
    """
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    a = np.clip(a, 0.0, 1.0)
    return 2.0 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


def _point_covers(polygon, lng, lat) -> np.ndarray:
    """Boundary-inclusive point-in-polygon, fast path via ``contains_xy``.

    Float lat/lng values never land exactly on the polygon boundary, so the
    strict interior test is equivalent in practice. Boundary inclusion is
    verified explicitly with scalar Shapely ``covers`` in the geography fixture
    (``checks.py``).
    """
    return contains_xy(polygon, lng, lat)


def prepare_boundary(repo_root: Path):
    """Load (or build once) the EPSG:4326 country polygon and its provenance manifest.

    Returns the single unioned polygon geometry.
    """
    gdir = geography_dir(repo_root)
    gj = gdir / "BR_Pais_2024_4326.geojson"
    manifest_path = gdir / "boundary_manifest.json"

    if gj.exists():
        poly = gpd.read_file(gj).geometry.union_all()
        return poly

    shp = gdir / "BR_Pais_2024.shp"
    if not shp.exists():
        raise FileNotFoundError(
            f"Missing {shp}. Stage the pinned IBGE BR_Pais_2024 archive under "
            f"{gdir} before running (feature_contract.md §4 boundary reference)."
        )

    gdf = gpd.read_file(shp)
    src_crs = str(gdf.crs)
    gdf = gdf.to_crs("EPSG:4326")
    # Union all country parts; dissolve retains supplied islands.
    gdf = gdf.dissolve()
    geom = gdf.geometry.iloc[0]

    if geom.is_empty:
        raise ValueError("IBGE boundary dissolved to empty geometry.")
    if not geom.is_valid:
        # Documented repair: zero buffer. Recorded in the manifest.
        geom = geom.buffer(0)
        if not geom.is_valid or geom.is_empty:
            raise ValueError("IBGE boundary invalid after zero-buffer repair.")

    out = gpd.GeoDataFrame({"geometry": [geom]}, crs="EPSG:4326")
    out.to_file(gj, driver="GeoJSON")

    manifest = {
        "source_url": (
            "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
            "malhas_municipais/municipio_2024/Brasil/BR_Pais_2024.zip"
        ),
        "product": "BR_Pais_2024 (IBGE national boundary)",
        "product_year": 2024,
        "retrieved_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "archive_sha256": _ARCHIVE_SHA256,
        "original_crs": src_crs,
        "target_crs": "EPSG:4326",
        "geometry_selection_rule": "union all country parts; retain supplied islands",
        "transformation": "read shapefile; to_crs EPSG:4326; dissolve to single geometry",
        "repair": "zero buffer applied only if source geometry was invalid",
        "library_versions": {
            "geopandas": gpd.__version__,
            "shapely": __import__("shapely").__version__,
            "pyproj": __import__("pyproj").__version__,
        },
        "derived_geometry_file": gj.name,
        "derived_geometry_sha256": _sha256_bytes(gj.read_bytes()),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    return geom


def build_zip_centroids(geolocation: pd.DataFrame, polygon):
    """8-step polygon-filtered ZIP/state median lookup (feature_contract.md §4).

    Returns ``(centroids, filter_counts)`` where centroids has one row per valid
    (zip, state) key and filter_counts tallies first-failure reasons.
    """
    df = geolocation.copy()
    df["_zip"] = df["geolocation_zip_code_prefix"].map(normalize_zip)
    df["_state"] = df["geolocation_state"].astype(str).str.strip().str.upper()
    lat = pd.to_numeric(df["geolocation_lat"], errors="coerce")
    lng = pd.to_numeric(df["geolocation_lng"], errors="coerce")

    reason = pd.Series(np.full(len(df), None, dtype=object), index=df.index)

    # 1. invalid key (state not in UF set, or zip not normalizable)
    invalid_key = df["_zip"].isna() | ~df["_state"].isin(VALID_STATES)
    reason[invalid_key] = "invalid_key"

    # 2. nonnumeric / nonfinite coordinates
    bad_coord = lat.isna() | lng.isna() | ~np.isfinite(lat) | ~np.isfinite(lng)
    reason[reason.isna() & bad_coord] = "invalid_coords"

    # 3. latitude range
    bad_lat = (lat < -90) | (lat > 90)
    reason[reason.isna() & bad_lat] = "lat_out_of_range"

    # 4. longitude range
    bad_lng = (lng < -180) | (lng > 180)
    reason[reason.isna() & bad_lng] = "lng_out_of_range"

    # 5. country polygon (boundary-inclusive)
    candidates = reason.isna()
    if candidates.any():
        cx = lng[candidates].to_numpy(dtype=float)
        cy = lat[candidates].to_numpy(dtype=float)
        inside = _point_covers(polygon, cx, cy)
        outside_mask = candidates.copy()
        outside_mask[candidates] = ~inside
        reason[outside_mask] = "outside_polygon"

    filter_counts = reason.value_counts(dropna=False).to_dict()

    # Group valid keys; equal weight per surviving preprocessed row.
    valid = df[reason != "invalid_key"].copy()
    valid["_accepted"] = reason[valid.index].isna().astype(int)
    acc = valid[valid["_accepted"] == 1]

    groups = (
        valid.groupby(["_zip", "_state"], sort=True)
        .agg(accepted_count=("_accepted", "sum"))
        .reset_index()
    )
    if len(acc):
        med = (
            acc.groupby(["_zip", "_state"], sort=True)
            .agg(median_lat=("geolocation_lat", "median"),
                 median_lng=("geolocation_lng", "median"))
            .reset_index()
        )
        groups = groups.merge(med, on=["_zip", "_state"], how="left")
    else:
        groups["median_lat"] = np.nan
        groups["median_lng"] = np.nan

    # Step 6: re-check median against the polygon; coordinate-wise medians can fall
    # outside nonconvex polygons.
    has_med = groups["median_lat"].notna()
    if has_med.any():
        mx = groups.loc[has_med, "median_lng"].to_numpy(dtype=float)
        my = groups.loc[has_med, "median_lat"].to_numpy(dtype=float)
        med_ok = _point_covers(polygon, mx, my)
        bad_idx = groups.index[has_med][~med_ok]
        groups.loc[bad_idx, "median_lat"] = np.nan
        groups.loc[bad_idx, "median_lng"] = np.nan

    def status():
        return np.where(
            groups["accepted_count"] == 0, "no_surviving_coordinates",
            np.where(groups["median_lat"].isna(), "median_outside_polygon", "resolved"))

    groups["resolution_status"] = status()
    groups = groups.rename(columns={"_zip": "zip_prefix", "_state": "state"})
    groups = groups.sort_values(["zip_prefix", "state"]).reset_index(drop=True)
    groups = groups[["zip_prefix", "state", "accepted_count", "median_lat",
                     "median_lng", "resolution_status"]]
    return groups, filter_counts


def compute_distance_features(orders: pd.DataFrame, customers: pd.DataFrame,
                              sellers: pd.DataFrame, order_seller_pairs: pd.DataFrame,
                              centroids: pd.DataFrame) -> pd.DataFrame:
    """Per-order ``distance_km_max``, ``distance_missing_fraction``, ``interstate_share``.

    Left-joins the composite (zip, state) centroid key; an unresolved coordinate
    (no surviving ZIP, out-of-country median, invalid state) nulls the whole
    order's maximum distance — never a partial maximum. Repeated seller items do
    not multiply distances because ``order_seller_pairs`` is already distinct.
    """
    cent = centroids[["zip_prefix", "state", "median_lat", "median_lng"]].rename(
        columns={"median_lat": "lat", "median_lng": "lng"})

    cust = customers.copy()
    cust["_zip"] = customers["customer_zip_code_prefix"].map(normalize_zip)
    cust["_state"] = customers["customer_state"].map(normalize_state)
    cust = cust.merge(cent, left_on=["_zip", "_state"],
                      right_on=["zip_prefix", "state"], how="left")

    sell = sellers.copy()
    sell["_zip"] = sellers["seller_zip_code_prefix"].map(normalize_zip)
    sell["_state"] = sellers["seller_state"].map(normalize_state)
    sell = sell.merge(cent, left_on=["_zip", "_state"],
                      right_on=["zip_prefix", "state"], how="left")

    osp = order_seller_pairs.merge(
        sell[["seller_id", "_state", "lat", "lng"]].rename(
            columns={"_state": "seller_state", "lat": "sell_lat", "lng": "sell_lng"}),
        on="seller_id", how="left",
    )
    # Distance features are defined over *distinct* sellers (feature_contract §4);
    # dedup defensively so repeated seller rows never inflate n_sellers.
    osp = osp.drop_duplicates(["order_id", "seller_id"])
    oc = orders[["order_id", "customer_id"]].drop_duplicates("order_id")
    custc = cust[["customer_id", "_state", "lat", "lng"]].rename(
        columns={"_state": "cust_state", "lat": "cust_lat", "lng": "cust_lng"})
    osp = osp.merge(oc, on="order_id", how="left").merge(custc, on="customer_id", how="left")

    osp["seller_resolved"] = osp["sell_lat"].notna() & osp["sell_lng"].notna()
    osp["cust_resolved"] = osp["cust_lat"].notna() & osp["cust_lng"].notna()
    osp["dist"] = haversine_km(osp["cust_lat"], osp["cust_lng"],
                               osp["sell_lat"], osp["sell_lng"]).to_numpy()
    osp["state_ok"] = (
        osp["cust_state"].notna() & (osp["cust_state"] != UNKNOWN)
        & osp["seller_state"].notna() & (osp["seller_state"] != UNKNOWN)
    )
    osp["is_interstate"] = osp["seller_state"] != osp["cust_state"]

    g = osp.groupby("order_id").agg(
        n_sellers=("seller_id", "size"),
        seller_unresolved=("seller_resolved", lambda s: int((~s).sum())),
        cust_resolved=("cust_resolved", "first"),
        all_state_ok=("state_ok", "all"),
        interstate_sum=("is_interstate", "sum"),
        dist_max=("dist", "max"),
    )

    n_sellers = g["n_sellers"].to_numpy(dtype=int)
    cust_resolved = g["cust_resolved"].to_numpy(dtype=bool)
    seller_unresolved = g["seller_unresolved"].to_numpy(dtype=int)
    dist_max = g["dist_max"].to_numpy(dtype=float)
    all_state_ok = g["all_state_ok"].to_numpy(dtype=bool)
    interstate_sum = g["interstate_sum"].to_numpy(dtype=float)

    # A missing customer coordinate makes every seller distance unresolved.
    dist_unresolved = np.where(cust_resolved, seller_unresolved, n_sellers)
    distance_km_max = np.where(dist_unresolved == 0, dist_max, np.nan)
    distance_missing_fraction = np.where(n_sellers > 0, dist_unresolved / n_sellers, np.nan)
    interstate_share = np.where(
        all_state_ok & (n_sellers > 0), interstate_sum / n_sellers, np.nan)

    return pd.DataFrame({
        "distance_km_max": distance_km_max,
        "distance_missing_fraction": distance_missing_fraction,
        "interstate_share": interstate_share,
    }, index=g.index)

```


### `src/features/aggregation.py`

Order-grain aggregation of items, payments and reviews (pure and deterministic — no item×payment×review Cartesian product).

```python
"""Order-grain aggregation of items, payments and reviews (feature_contract v1.1).

Every function is pure and deterministic: it consumes preprocessed tables and
returns per-``order_id`` frames. No item x payment x review Cartesian product is
ever formed — each fact table is collapsed to the order grain independently.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .contract import UNKNOWN, USABLE_PAYMENT_TYPES


# ---------------------------------------------------------------------------
# Items
# ---------------------------------------------------------------------------

def resolve_english_category(products: pd.DataFrame, translation: pd.DataFrame) -> pd.DataFrame:
    """Attach a resolved English category; null for missing/untranslated names.

    The English label comes from ``product_category_name_translation.csv`` keyed
    on the Portuguese ``product_category_name``. Categories absent from the
    translation table stay unresolved (counted in ``category_missing_fraction``).
    """
    t = translation.rename(columns={
        "product_category_name": "_cat_pt",
        "product_category_name_english": "english_category",
    })
    prod = products.merge(
        t[["_cat_pt", "english_category"]],
        left_on="product_category_name", right_on="_cat_pt", how="left",
    ).drop(columns=["_cat_pt"])
    return prod


def aggregate_items(products_with_cat: pd.DataFrame,
                    items: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Collapse items to the order grain.

    Returns ``(item_agg, primary_rows, order_seller_pairs)`` where:
    - ``item_agg`` is indexed by ``order_id`` and carries every item-derived
      feature plus the primary item's resolved attributes;
    - ``primary_rows`` is the deterministic primary item per order;
    - ``order_seller_pairs`` is the distinct (order_id, seller_id) set for
      distance/interstate computation.
    """
    it = items.merge(
        products_with_cat[
            ["product_id", "english_category", "product_weight_g",
             "product_length_cm", "product_height_cm", "product_width_cm"]
        ],
        on="product_id", how="left",
    )

    price = pd.to_numeric(it["price"], errors="coerce")
    freight = pd.to_numeric(it["freight_value"], errors="coerce")
    weight = pd.to_numeric(it["product_weight_g"], errors="coerce")
    ln = pd.to_numeric(it["product_length_cm"], errors="coerce")
    ht = pd.to_numeric(it["product_height_cm"], errors="coerce")
    wd = pd.to_numeric(it["product_width_cm"], errors="coerce")

    it["_price_ok"] = price.notna() & np.isfinite(price) & (price >= 0)
    it["_freight_ok"] = freight.notna() & np.isfinite(freight) & (freight >= 0)
    it["_weight_ok"] = weight.notna() & np.isfinite(weight) & (weight > 0)
    it["_vol_ok"] = (ln.notna() & np.isfinite(ln) & (ln > 0)
                     & ht.notna() & np.isfinite(ht) & (ht > 0)
                     & wd.notna() & np.isfinite(wd) & (wd > 0))
    it["_cat_ok"] = it["english_category"].notna()

    it["_weight"] = weight.where(it["_weight_ok"])
    it["_volume"] = (ln * ht * wd).where(it["_vol_ok"])

    agg = it.groupby("order_id").agg(
        n_items=("order_item_id", "size"),
        n_products=("product_id", "nunique"),
        n_sellers=("seller_id", "nunique"),
        n_categories=("english_category", "nunique"),
        price_invalid=("_price_ok", lambda s: int((~s).sum())),
        freight_invalid=("_freight_ok", lambda s: int((~s).sum())),
        weight_invalid=("_weight_ok", lambda s: int((~s).sum())),
        volume_invalid=("_vol_ok", lambda s: int((~s).sum())),
        category_missing=("_cat_ok", lambda s: int((~s).sum())),
        seller_missing=("seller_id", lambda s: int(s.isna().sum())),
        total_price_sum=("price", "sum"),
        total_freight_sum=("freight_value", "sum"),
        weight_sum=("_weight", "sum"),
        volume_sum=("_volume", "sum"),
    )

    n = agg["n_items"].astype(float)

    agg["total_price"] = np.where(agg["price_invalid"] == 0, agg["total_price_sum"], np.nan)
    agg["total_freight"] = np.where(agg["freight_invalid"] == 0, agg["total_freight_sum"], np.nan)
    agg["total_weight_g"] = np.where(agg["weight_invalid"] == 0, agg["weight_sum"], np.nan)
    agg["total_volume_cm3"] = np.where(agg["volume_invalid"] == 0, agg["volume_sum"], np.nan)

    agg["weight_missing_fraction"] = agg["weight_invalid"] / n
    agg["volume_missing_fraction"] = agg["volume_invalid"] / n
    agg["category_missing_fraction"] = agg["category_missing"] / n

    agg["freight_ratio"] = np.where(
        (agg["total_price"].isna()) | (agg["total_price"] <= 0) | (agg["total_freight"].isna()),
        np.nan,
        agg["total_freight"] / agg["total_price"],
    )
    agg["has_items"] = (agg["n_items"] > 0).astype(int)
    agg["freight_ratio_missing"] = agg["freight_ratio"].isna().astype(int)
    agg["seller_id_missing"] = (agg["seller_missing"] > 0).astype(int)

    # Deterministic primary item: price desc (missing last), order_item_id asc.
    it_sorted = it.sort_values(
        ["order_id", "price", "order_item_id"],
        ascending=[True, False, True], na_position="last", kind="mergesort",
    )
    primary = it_sorted.drop_duplicates("order_id", keep="first").set_index("order_id")
    primary = primary[["seller_id", "english_category"]].rename(
        columns={"seller_id": "primary_seller_id",
                 "english_category": "primary_category"})

    order_seller_pairs = (
        it[["order_id", "seller_id"]].dropna(subset=["seller_id"])
        .drop_duplicates().reset_index(drop=True)
    )

    return agg, primary, order_seller_pairs


# ---------------------------------------------------------------------------
# Payments
# ---------------------------------------------------------------------------

def aggregate_payments(payments: pd.DataFrame) -> pd.DataFrame:
    """Collapse payments to the order grain.

    Usable rows = usable type (credit_card/boleto/voucher/debit_card, normalized)
    AND finite positive payment_value. ``payment_installments_max`` uses only
    positive, finite installments.
    """
    p = payments.copy()
    p["_type"] = p["payment_type"].map(
        lambda v: str(v).strip().lower()
        if isinstance(v, str) else (str(v).strip().lower() if v is not None else None)
    )
    p.loc[~p["_type"].isin(USABLE_PAYMENT_TYPES), "_type"] = None

    val = pd.to_numeric(p["payment_value"], errors="coerce")
    inst = pd.to_numeric(p["payment_installments"], errors="coerce")

    p["_usable"] = p["_type"].notna() & val.notna() & np.isfinite(val) & (val > 0)
    p["_val"] = val.where(p["_usable"])
    p["_inst"] = inst.where(p["_usable"] & inst.notna() & np.isfinite(inst) & (inst > 0))

    usable = p[p["_usable"]]

    totals = (
        usable.groupby(["order_id", "_type"], as_index=False)["_val"].sum()
        .rename(columns={"_val": "_total"})
    )
    totals = totals.sort_values(
        ["order_id", "_total", "_type"], ascending=[True, False, True], kind="mergesort"
    )
    primary = totals.drop_duplicates("order_id", keep="first")[["order_id", "_type"]]

    out = pd.DataFrame({"order_id": payments["order_id"].unique()})

    if len(usable):
        n_methods = usable.groupby("order_id")["_type"].nunique().rename("n_payment_methods")
        inst_max = usable.groupby("order_id")["_inst"].max().rename("payment_installments_max")
        out = out.merge(n_methods, on="order_id", how="left")
        out = out.merge(inst_max, on="order_id", how="left")
    else:
        out["n_payment_methods"] = np.nan
        out["payment_installments_max"] = np.nan

    out = out.merge(
        primary.rename(columns={"_type": "primary_payment_type"}), on="order_id", how="left"
    )
    out["n_payment_methods"] = out["n_payment_methods"].fillna(0).astype(int)
    out["primary_payment_type"] = out["primary_payment_type"].fillna(UNKNOWN)
    out["payment_missing"] = (out["primary_payment_type"] == UNKNOWN).astype(int)
    return out.set_index("order_id")


# ---------------------------------------------------------------------------
# Reviews
# ---------------------------------------------------------------------------

def aggregate_reviews(reviews: pd.DataFrame) -> pd.DataFrame:
    """Collapse reviews to the order grain.

    A valid score is an integer in 1..5. ``review_latest_observed_at`` is the max
    valid answer timestamp; a valid score with a missing answer date still feeds
    ``review_score_min`` but not the observation timestamp.
    """
    r = reviews.copy()
    score = pd.to_numeric(r["review_score"], errors="coerce")
    valid = score.notna() & np.isfinite(score) & (score % 1 == 0) & (score >= 1) & (score <= 5)
    ans = pd.to_datetime(r["review_answer_timestamp"], errors="coerce", utc=False)

    r["_score"] = score.where(valid)
    r["_ans_contrib"] = ans.where(valid)

    out = r.groupby("order_id").agg(
        review_count=("review_id", "size"),
        review_score_min=("_score", "min"),
        review_latest_observed_at=("_ans_contrib", "max"),
    )

    det = pd.Series(pd.NA, index=out.index, dtype="Int64")
    det[out["review_score_min"] <= 2] = 1
    det[out["review_score_min"] >= 3] = 0
    out["is_detractor"] = det

    rmin = pd.Series(pd.NA, index=out.index, dtype="Int64")
    rmin[out["review_score_min"].notna()] = out.loc[out["review_score_min"].notna(), "review_score_min"].astype("Int64")
    out["review_score_min"] = rmin
    return out

```


### `src/features/checks.py`

Thirteen synthetic fixture checks that exercise the exact contract rules (geometry, tie-breaking, missingness) without touching real data.

```python
"""Small, deterministic fixture checks for feature_contract.md §9 (Phase A).

Each check returns ``{"id", "status", "detail"}`` where status is ``pass`` or
``fail``. These exercise the exact contract rules with synthetic inputs so that a
regression in geometry, aggregation or tie-breaking is caught without touching
the real preprocessed data.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon

from .aggregation import aggregate_items, aggregate_payments, aggregate_reviews
from .contract import PREDICTOR_ALLOWLIST, UNKNOWN, VALID_STATES
from .geography import (
    build_zip_centroids,
    compute_distance_features,
    haversine_km,
    normalize_zip,
    _point_covers,
)


def _ok(detail: str) -> dict:
    return {"status": "pass", "detail": detail}


def _fail(detail: str) -> dict:
    return {"status": "fail", "detail": detail}


def _close(a, b, tol=1e-6) -> bool:
    return abs(float(a) - float(b)) <= tol


# ---------------------------------------------------------------------------
# Geography fixtures
# ---------------------------------------------------------------------------

def check_geography(polygon) -> dict:
    """Reject a European point and a bounding-box-but-not-Brazil point; accept a
    boundary point (scalar Shapely ``covers``, boundary-inclusive)."""
    paris = _point_covers(polygon, np.array([2.3522]), np.array([48.8566]))
    if bool(paris[0]):
        return _fail("Paris (48.86, 2.35) was accepted inside the Brazil polygon")

    ocean = _point_covers(polygon, np.array([-40.0]), np.array([-28.0]))
    if bool(ocean[0]):
        return _fail("South-Atlantic point (-28, -40) inside the bbox was accepted")

    boundary = polygon.boundary
    if boundary.geom_type == "MultiLineString":
        pt = boundary.geoms[0].coords[0]
    else:
        pt = boundary.coords[0]
    if not polygon.covers(Point(pt[0], pt[1])):
        return _fail("a point on the polygon boundary was rejected by covers")
    return _ok("European/bbox-foreign points rejected; boundary point accepted via covers")


def check_mixed_validity_zip(polygon) -> dict:
    """A mixed-validity ZIP uses only accepted (in-polygon) coordinates."""
    sq = Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])  # lat 0..10, lng 0..10
    geo = pd.DataFrame({
        "geolocation_zip_code_prefix": [11111, 11111, 11111, 11111],
        "geolocation_lat": [1.0, 2.0, 3.0, 50.0],  # 50 is outside
        "geolocation_lng": [5.0, 5.0, 5.0, 5.0],
        "geolocation_state": ["XX", "XX", "XX", "XX"],  # XX not a valid UF -> invalid key
    })
    # Use a valid UF so the key is valid; the square is just a toy polygon.
    geo["geolocation_state"] = "SP"
    centroids, counts = build_zip_centroids(geo, sq)
    row = centroids[centroids["zip_prefix"] == "11111"].iloc[0]
    if not _close(row["median_lat"], 2.0):
        return _fail(f"median lat {row['median_lat']} != 2.0 (outside point leaked)")
    if int(row["accepted_count"]) != 3:
        return _fail(f"accepted_count {row['accepted_count']} != 3")
    return _ok("mixed-validity ZIP median uses only the 3 in-polygon points")


def check_all_rejected_zip(polygon) -> dict:
    """An all-rejected ZIP stays unresolved with null coordinates."""
    sq = Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])
    geo = pd.DataFrame({
        "geolocation_zip_code_prefix": [22222, 22222],
        "geolocation_lat": [50.0, 60.0],
        "geolocation_lng": [50.0, 60.0],
        "geolocation_state": ["SP", "SP"],
    })
    centroids, counts = build_zip_centroids(geo, sq)
    row = centroids[centroids["zip_prefix"] == "22222"].iloc[0]
    if row["resolution_status"] != "no_surviving_coordinates":
        return _fail(f"status {row['resolution_status']} != no_surviving_coordinates")
    if not (pd.isna(row["median_lat"]) and pd.isna(row["median_lng"])):
        return _fail("all-rejected ZIP produced coordinates")
    if int(row["accepted_count"]) != 0:
        return _fail("accepted_count != 0 for all-rejected ZIP")
    return _ok("all-rejected ZIP remains unresolved with null coordinates")


def check_zip_normalization() -> dict:
    """Integer/digit prefixes zero-pad; malformed forms reject."""
    if normalize_zip(1234) != "01234":
        return _fail("normalize_zip(1234) != '01234'")
    if normalize_zip("42") != "00042":
        return _fail("normalize_zip('42') != '00042'")
    if normalize_zip("abc") is not None:
        return _fail("normalize_zip('abc') should be None")
    if normalize_zip(123456) is not None:
        return _fail("normalize_zip(123456) should be None")
    if normalize_zip(None) is not None:
        return _fail("normalize_zip(None) should be None")
    return _ok("ZIP prefix normalization zero-pads and rejects malformed forms")


# ---------------------------------------------------------------------------
# Distance fixtures
# ---------------------------------------------------------------------------

def check_haversine() -> dict:
    if not _close(float(haversine_km(0.0, 0.0, 0.0, 0.0)), 0.0):
        return _fail("coincident points did not give zero distance")
    # Known: ~1 degree latitude ~ 111.19 km.
    d = float(haversine_km(0.0, 0.0, 1.0, 0.0))
    if not _close(d, 111.19, tol=0.1):
        return _fail(f"1-degree latitude distance {d} != ~111.19 km")
    return _ok("haversine: coincident=0; 1-degree latitude ~111.19 km")


def check_distance_aggregation() -> dict:
    """Single/multiple/duplicate sellers and a missing-coordinate seller."""
    centroids = pd.DataFrame({
        "zip_prefix": ["01000", "02000", "03000"],
        "state": ["SP", "RJ", "MG"],
        "accepted_count": [1, 1, 1],
        "median_lat": [-23.55, -22.90, -19.92],
        "median_lng": [-46.63, -43.20, -43.94],
        "resolution_status": ["resolved", "resolved", "resolved"],
    })
    customers = pd.DataFrame({
        "customer_id": ["c1"], "customer_unique_id": ["cu1"],
        "customer_zip_code_prefix": [1000], "customer_state": ["SP"],
    })
    sellers = pd.DataFrame({
        "seller_id": ["s1", "s2", "s3"],
        "seller_zip_code_prefix": [2000, 3000, 9999],
        "seller_state": ["RJ", "MG", "AC"],
    })
    orders = pd.DataFrame({"order_id": ["o1"], "customer_id": ["c1"]})
    pairs = pd.DataFrame({
        "order_id": ["o1", "o1", "o1", "o1"],
        "seller_id": ["s1", "s2", "s1", "s3"],  # s1 duplicated
    })

    d_s1 = float(haversine_km(-23.55, -46.63, -22.90, -43.20))
    d_s2 = float(haversine_km(-23.55, -46.63, -19.92, -43.94))

    # s3 has zip 9999 with no centroid -> unresolved -> distance null.
    res = compute_distance_features(orders, customers, sellers, pairs, centroids)
    row = res.loc["o1"]
    if not pd.isna(row["distance_km_max"]):
        return _fail("distance_km_max should be null when a seller coordinate is unresolved")
    if not _close(row["distance_missing_fraction"], 1.0 / 3.0):
        return _fail(f"distance_missing_fraction {row['distance_missing_fraction']} != 1/3")
    if not _close(row["interstate_share"], 1.0):
        return _fail("interstate_share should be 1.0 (all sellers out of SP)")

    # Without the unresolved seller, the max is over {s1, s2} and dup s1 is ignored.
    pairs_ok = pd.DataFrame({"order_id": ["o1", "o1", "o1"], "seller_id": ["s1", "s2", "s1"]})
    res2 = compute_distance_features(orders, customers, sellers, pairs_ok, centroids)
    row2 = res2.loc["o1"]
    if not _close(row2["distance_km_max"], max(d_s1, d_s2)):
        return _fail(f"distance_km_max {row2['distance_km_max']} != max({d_s1},{d_s2})")
    return _ok("distance: single/dup/max semantics and null-on-missing verified")


# ---------------------------------------------------------------------------
# Physical / aggregation fixtures
# ---------------------------------------------------------------------------

def _mini_products() -> pd.DataFrame:
    return pd.DataFrame({
        "product_id": ["p_ok", "p_badw", "p_badv"],
        "product_category_name": ["cat_a", "cat_b", "cat_c"],
        "english_category": ["Cat A", "Cat B", None],
        "product_weight_g": [100.0, np.nan, 50.0],
        "product_length_cm": [10.0, 10.0, np.nan],
        "product_height_cm": [10.0, 10.0, 10.0],
        "product_width_cm": [10.0, 10.0, 10.0],
    })


def check_physical_totals() -> dict:
    """Two items, one missing weight -> null total weight + 0.5 fraction; both
    volumes valid -> total volume is the plain sum."""
    prod = _mini_products()
    items = pd.DataFrame({
        "order_id": ["o1", "o1"],
        "order_item_id": [1, 2],
        "product_id": ["p_ok", "p_badw"],
        "seller_id": ["s1", "s1"],
        "price": [10.0, 20.0],
        "freight_value": [5.0, 5.0],
    })
    agg, primary, pairs = aggregate_items(prod, items)
    row = agg.loc["o1"]
    if not pd.isna(row["total_weight_g"]):
        return _fail("total_weight_g should be null with one missing weight")
    if not _close(row["weight_missing_fraction"], 0.5):
        return _fail("weight_missing_fraction != 0.5")
    if not _close(row["total_volume_cm3"], 2000.0):
        return _fail(f"total_volume_cm3 {row['total_volume_cm3']} != 2000 (both volumes valid)")
    if not _close(row["volume_missing_fraction"], 0.0):
        return _fail("volume_missing_fraction != 0.0")
    return _ok("physical totals null on missing measurement; fractions correct")


def check_zero_price() -> dict:
    """A zero price preserves the row and yields a null freight ratio."""
    prod = _mini_products()
    items = pd.DataFrame({
        "order_id": ["o1"], "order_item_id": [1], "product_id": ["p_ok"],
        "seller_id": ["s1"], "price": [0.0], "freight_value": [5.0],
    })
    agg, primary, pairs = aggregate_items(prod, items)
    row = agg.loc["o1"]
    if row["n_items"] != 1 or row["has_items"] != 1:
        return _fail("zero-price order dropped or has_items=0")
    if not _close(row["total_price"], 0.0):
        return _fail("total_price should be 0.0")
    if not pd.isna(row["freight_ratio"]) or row["freight_ratio_missing"] != 1:
        return _fail("freight_ratio should be null with missing flag 1")
    return _ok("zero price preserved; freight ratio null")


def check_primary_item_tie() -> dict:
    """Equal prices -> smallest numeric order_item_id wins."""
    prod = _mini_products()
    items = pd.DataFrame({
        "order_id": ["o1", "o1"],
        "order_item_id": [5, 2],
        "product_id": ["p_ok", "p_badw"],
        "seller_id": ["s1", "s2"],
        "price": [10.0, 10.0],
        "freight_value": [1.0, 1.0],
    })
    agg, primary, pairs = aggregate_items(prod, items)
    row = primary.loc["o1"]
    if row["primary_seller_id"] != "s2":
        return _fail(f"primary seller {row['primary_seller_id']} != s2 (smallest item id)")
    return _ok("primary-item tie resolved by smallest order_item_id")


def check_payment_tie() -> dict:
    """Equal type totals -> lexicographically smallest type wins."""
    payments = pd.DataFrame({
        "order_id": ["o1", "o1", "o1", "o1"],
        "payment_sequential": [1, 2, 3, 4],
        "payment_type": ["credit_card", "boleto", "credit_card", "boleto"],
        "payment_installments": [1, 1, 1, 1],
        "payment_value": [50.0, 50.0, 50.0, 50.0],
    })
    out = aggregate_payments(payments)
    row = out.loc["o1"]
    if row["primary_payment_type"] != "boleto":
        return _fail(f"tie primary type {row['primary_payment_type']} != boleto")
    if row["n_payment_methods"] != 2 or row["payment_missing"] != 0:
        return _fail("n_payment_methods/payment_missing wrong for two-method order")
    return _ok("payment-type tie resolved lexicographically (boleto < credit_card)")


def check_absent_payment_and_unknown_category() -> dict:
    """Absent payments -> Unknown type, payment_missing=1; untranslated category -> Unknown."""
    prod = pd.DataFrame({
        "product_id": ["p_x"],
        "product_category_name": ["cat_c"],
        "english_category": [None],
        "product_weight_g": [10.0],
        "product_length_cm": [10.0], "product_height_cm": [10.0], "product_width_cm": [10.0],
    })
    items = pd.DataFrame({
        "order_id": ["o1"], "order_item_id": [1], "product_id": ["p_x"],
        "seller_id": ["s1"], "price": [10.0], "freight_value": [1.0],
    })
    agg, primary, pairs = aggregate_items(prod, items)
    row = primary.loc["o1"]
    if not pd.isna(row["primary_category"]):
        return _fail("untranslated category should be unresolved at aggregation")

    empty_pay = pd.DataFrame(columns=["order_id", "payment_sequential",
                                      "payment_type", "payment_installments", "payment_value"])
    out = aggregate_payments(empty_pay)
    # no rows -> function returns frame over payments.order_id.unique() (empty)
    if len(out) != 0:
        return _fail("aggregate_payments of empty input should return empty frame")
    return _ok("untranslated category unresolved; empty payments handled")


def check_review_target() -> dict:
    """Missing review -> null detractor; valid score -> correct detractor mapping."""
    reviews = pd.DataFrame({
        "review_id": ["r1", "r2", "r3"],
        "order_id": ["o1", "o2", "o3"],
        "review_score": [1, 5, 3],
        "review_answer_timestamp": ["2018-01-01", "2018-01-02", "2018-01-03"],
    })
    out = aggregate_reviews(reviews)
    if int(out.loc["o1", "is_detractor"]) != 1:
        return _fail("score 1 should map to is_detractor=1")
    if int(out.loc["o2", "is_detractor"]) != 0:
        return _fail("score 5 should map to is_detractor=0")
    if int(out.loc["o3", "is_detractor"]) != 0:
        return _fail("score 3 should map to is_detractor=0")
    return _ok("review target: min-score detractor mapping verified")


# ---------------------------------------------------------------------------
# Allowlist separation
# ---------------------------------------------------------------------------

def check_allowlist_separation(export_columns: list[str]) -> dict:
    """The 27-column allowlist must not intersect targets/IDs/outcome/eligibility/audit."""
    forbidden = {
        "order_id", "customer_id", "customer_unique_id",
        "prediction_timestamp", "order_status",
        "lead_days", "review_score_min", "is_detractor",
        "eligible_regression", "eligible_classification",
        "regression_exclusion_reason", "classification_exclusion_reason",
        "regression_label_available_at", "review_latest_observed_at",
        "review_count", "seller_id_missing",
        "split_assignment", "validation_fold",
    }
    overlap = set(PREDICTOR_ALLOWLIST) & forbidden
    if overlap:
        return _fail(f"allowlist intersects forbidden columns: {sorted(overlap)}")
    missing = [c for c in PREDICTOR_ALLOWLIST if c not in export_columns]
    if missing:
        return _fail(f"allowlist columns missing from export: {missing}")
    if len(PREDICTOR_ALLOWLIST) != 27:
        return _fail(f"allowlist has {len(PREDICTOR_ALLOWLIST)} columns, expected 27")
    return _ok("allowlist disjoint from targets/IDs/outcome/audit; 27 columns exported")


# ---------------------------------------------------------------------------
# Runner
# ---------------------------------------------------------------------------

def run_fixture_checks(polygon, export_columns: list[str]) -> list[dict]:
    """Run all Phase-A fixtures; returns a list of {id, status, detail}."""
    results = []
    for fid, fn in [
        ("geography_reject_foreign_accept_boundary", lambda: check_geography(polygon)),
        ("geography_mixed_validity_zip", lambda: check_mixed_validity_zip(polygon)),
        ("geography_all_rejected_zip", lambda: check_all_rejected_zip(polygon)),
        ("zip_normalization", check_zip_normalization),
        ("haversine", check_haversine),
        ("distance_aggregation", check_distance_aggregation),
        ("physical_totals", check_physical_totals),
        ("zero_price", check_zero_price),
        ("primary_item_tie", check_primary_item_tie),
        ("payment_tie", check_payment_tie),
        ("absent_payment_unknown_category", check_absent_payment_and_unknown_category),
        ("review_target", check_review_target),
        ("allowlist_separation", lambda: check_allowlist_separation(export_columns)),
    ]:
        try:
            res = fn()
        except Exception as exc:  # noqa: BLE001 - surface as a failed check
            res = _fail(f"exception: {exc!r}")
        results.append({"id": fid, **res})
    return results

```


### `src/features/build_features.py`

Deterministic base-table builder: loaders, assembly, targets, eligibility, export and the §9 acceptance checks / quality report.

```python
"""Deterministic base-table builder (feature_contract.md v1.1, Phase A).

Entrypoint: ``python -m src.features.build_features``

Reads ``data/preprocessed/*.csv`` (plus the pinned IBGE boundary under
``docs/references/geography/``) and writes, under ``data/business/ml/``:

    orders_ml_features.csv   one row per source order
    feature_schema.json      versioned column roles/types/units/formulas
    dataset_manifest.json    input hashes, config, row counts, content checksum
    zip_centroids.csv        polygon-filtered ZIP/state median lookup
    quality_report.json      contract checks, missingness, cardinalities, cohorts

No imputation, scaling, encoding or learned transform happens here; those are
training-fold operations. Run ``python -m src.features.create_splits`` afterwards
for the development/holdout and CV manifests.
"""
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from src.common.loaders import sha256_file

from .aggregation import (
    aggregate_items,
    aggregate_payments,
    aggregate_reviews,
    resolve_english_category,
)
from .checks import run_fixture_checks
from .contract import (
    BASE_COLUMN_ORDER,
    CATEGORICAL_COLUMNS,
    FEATURE_SCHEMA,
    NUMERIC_COLUMNS,
    PREDICTOR_ALLOWLIST,
    UNKNOWN,
    normalize_state,
)
from .geography import (
    build_zip_centroids,
    compute_distance_features,
    normalize_zip,
    prepare_boundary,
)

FEATURE_VERSION = "v1.1"
SPLIT_VERSION = "v1.0"
CONTRACT_VERSION = "v1.1"

# ---------------------------------------------------------------------------
# Loaders (explicit dtypes + date parsing for determinism)
# ---------------------------------------------------------------------------

def _load_orders(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "olist_orders_dataset.csv",
        usecols=["order_id", "customer_id", "order_status",
                 "order_purchase_timestamp", "order_delivered_customer_date"],
        dtype={"order_id": "str", "customer_id": "str", "order_status": "str"},
        parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"],
    )


def _load_items(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "olist_order_items_dataset.csv",
        usecols=["order_id", "order_item_id", "product_id", "seller_id",
                 "price", "freight_value"],
        dtype={"order_id": "str", "product_id": "str", "seller_id": "str",
               "order_item_id": "int64", "price": "float64", "freight_value": "float64"},
    )


def _load_payments(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "olist_order_payments_dataset.csv",
        usecols=["order_id", "payment_sequential", "payment_type",
                 "payment_installments", "payment_value"],
        dtype={"order_id": "str", "payment_type": "str", "payment_sequential": "int64",
               "payment_installments": "float64", "payment_value": "float64"},
    )


def _load_reviews(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "olist_order_reviews_dataset.csv",
        usecols=["review_id", "order_id", "review_score", "review_answer_timestamp"],
        dtype={"review_id": "str", "order_id": "str", "review_score": "float64"},
        parse_dates=["review_answer_timestamp"],
    )


def _load_customers(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "olist_customers_dataset.csv",
        usecols=["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                 "customer_state"],
        dtype={"customer_id": "str", "customer_unique_id": "str", "customer_state": "str",
               "customer_zip_code_prefix": "int64"},
    )


def _load_products(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "olist_products_dataset.csv",
        usecols=["product_id", "product_category_name", "product_weight_g",
                 "product_length_cm", "product_height_cm", "product_width_cm"],
        dtype={"product_id": "str", "product_category_name": "str",
               "product_weight_g": "float64", "product_length_cm": "float64",
               "product_height_cm": "float64", "product_width_cm": "float64"},
    )


def _load_sellers(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "olist_sellers_dataset.csv",
        usecols=["seller_id", "seller_zip_code_prefix", "seller_state"],
        dtype={"seller_id": "str", "seller_state": "str", "seller_zip_code_prefix": "int64"},
    )


def _load_geolocation(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "olist_geolocation_dataset.csv",
        usecols=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
                 "geolocation_state"],
        dtype={"geolocation_state": "str", "geolocation_zip_code_prefix": "int64",
               "geolocation_lat": "float64", "geolocation_lng": "float64"},
    )


def _load_translation(pre: Path) -> pd.DataFrame:
    return pd.read_csv(
        pre / "product_category_name_translation.csv",
        dtype={"product_category_name": "str", "product_category_name_english": "str"},
    )


def _assert_unique(df: pd.DataFrame, cols, label: str) -> None:
    sub = df[cols]
    if sub.duplicated().any():
        raise ValueError(f"{label}: duplicate key {list(cols)} found")
    if sub.isna().any().any():
        raise ValueError(f"{label}: null key {list(cols)} found")


# ---------------------------------------------------------------------------
# Reason-code assembly
# ---------------------------------------------------------------------------

def _join_reasons(ordered: list[tuple[str, np.ndarray]]) -> list[str]:
    parts = [np.where(mask, label, "") for label, mask in ordered]
    return [";".join(x for x in row if x) for row in zip(*parts)]


# ---------------------------------------------------------------------------
# Per-feature schema enrichment (source columns, grain, availability)
# ---------------------------------------------------------------------------

_FEATURE_META: dict[str, dict] = {
    "n_items": {"source_columns": ["order_item_id"], "grain": "order"},
    "has_items": {"source_columns": ["n_items"], "grain": "order"},
    "n_products": {"source_columns": ["product_id"], "grain": "order"},
    "n_sellers": {"source_columns": ["seller_id"], "grain": "order"},
    "n_categories": {"source_columns": ["english_category"], "grain": "order"},
    "total_price": {"source_columns": ["price"], "grain": "order"},
    "total_freight": {"source_columns": ["freight_value"], "grain": "order"},
    "freight_ratio": {"source_columns": ["total_freight", "total_price"], "grain": "order"},
    "freight_ratio_missing": {"source_columns": ["freight_ratio"], "grain": "order"},
    "total_weight_g": {"source_columns": ["product_weight_g"], "grain": "order"},
    "total_volume_cm3": {"source_columns": ["product_length_cm", "product_height_cm", "product_width_cm"], "grain": "order"},
    "weight_missing_fraction": {"source_columns": ["product_weight_g"], "grain": "order"},
    "volume_missing_fraction": {"source_columns": ["product_length_cm", "product_height_cm", "product_width_cm"], "grain": "order"},
    "primary_category": {"source_columns": ["english_category"], "grain": "order (primary item)"},
    "category_missing_fraction": {"source_columns": ["english_category"], "grain": "order"},
    "customer_state": {"source_columns": ["customer_state"], "grain": "customer"},
    "primary_seller_state": {"source_columns": ["seller_state"], "grain": "order (primary seller)"},
    "distance_km_max": {"source_columns": ["geolocation_lat", "geolocation_lng", "customer_zip_code_prefix", "customer_state", "seller_zip_code_prefix", "seller_state"], "grain": "order (distinct sellers)"},
    "distance_missing_fraction": {"source_columns": ["distance_km_max"], "grain": "order (distinct sellers)"},
    "interstate_share": {"source_columns": ["customer_state", "seller_state"], "grain": "order (distinct sellers)"},
    "purchase_month": {"source_columns": ["order_purchase_timestamp"], "grain": "order"},
    "purchase_dayofweek": {"source_columns": ["order_purchase_timestamp"], "grain": "order"},
    "purchase_hour": {"source_columns": ["order_purchase_timestamp"], "grain": "order"},
    "primary_payment_type": {"source_columns": ["payment_type", "payment_value"], "grain": "order"},
    "payment_installments_max": {"source_columns": ["payment_installments"], "grain": "order"},
    "n_payment_methods": {"source_columns": ["payment_type"], "grain": "order"},
    "payment_missing": {"source_columns": ["payment_type", "payment_value"], "grain": "order"},
}

_AVAILABILITY_DEFAULT = (
    "checkout-complete simulation: item/payment/catalogue snapshot availability assumed; "
    "payments have no event timestamps and catalogue attributes have no historical versions"
)
_AVAILABILITY = {
    "distance_km_max": "static IBGE BR_Pais_2024 boundary (2016-2018 orders); label-independent reference",
    "distance_missing_fraction": "static IBGE BR_Pais_2024 boundary (2016-2018 orders); label-independent reference",
    "purchase_month": "source clock convention; no timezone conversion",
    "purchase_dayofweek": "source clock convention; no timezone conversion",
    "purchase_hour": "source clock convention; no timezone conversion",
}


def _enriched_schema() -> dict:
    cols = []
    for c in FEATURE_SCHEMA:
        entry = dict(c)
        if c["role"] == "feature":
            meta = _FEATURE_META[c["name"]]
            entry["source_columns"] = meta["source_columns"]
            entry["aggregation_grain"] = meta["grain"]
            entry["availability_assumption"] = _AVAILABILITY.get(c["name"], _AVAILABILITY_DEFAULT)
        cols.append(entry)
    return {
        "version": FEATURE_VERSION,
        "contract_version": CONTRACT_VERSION,
        "predictor_allowlist": PREDICTOR_ALLOWLIST,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "numeric_columns": NUMERIC_COLUMNS,
        "columns": cols,
    }


# ---------------------------------------------------------------------------
# Main build
# ---------------------------------------------------------------------------

def build(repo_root: Path) -> dict:
    pre = repo_root / "data" / "preprocessed"
    ml = repo_root / "data" / "business" / "ml"
    ml.mkdir(parents=True, exist_ok=True)

    orders = _load_orders(pre)
    items = _load_items(pre)
    payments = _load_payments(pre)
    reviews = _load_reviews(pre)
    customers = _load_customers(pre)
    products = _load_products(pre)
    sellers = _load_sellers(pre)
    geolocation = _load_geolocation(pre)
    translation = _load_translation(pre)

    # Key uniqueness (fail, never silently drop).
    _assert_unique(orders, "order_id", "orders")
    _assert_unique(customers, "customer_id", "customers")
    _assert_unique(products, "product_id", "products")
    _assert_unique(sellers, "seller_id", "sellers")
    _assert_unique(translation, "product_category_name", "translation")
    _assert_unique(items, ["order_id", "order_item_id"], "items")
    _assert_unique(payments, ["order_id", "payment_sequential"], "payments")

    # Boundary + ZIP/state median lookup.
    polygon = prepare_boundary(repo_root)
    centroids, geo_filter_counts = build_zip_centroids(geolocation, polygon)
    centroids.to_csv(ml / "zip_centroids.csv", index=False)

    # Categories, then order-grain aggregation (items / payments / reviews).
    products_cat = resolve_english_category(products, translation)
    item_agg, primary, order_seller_pairs = aggregate_items(products_cat, items)
    pay_agg = aggregate_payments(payments)
    rev_agg = aggregate_reviews(reviews)

    # Geography (distance/interstate) over distinct seller pairs.
    geo = compute_distance_features(orders, customers, sellers, order_seller_pairs, centroids)

    # Assemble one row per order.
    base = orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
    base = base.merge(item_agg, left_on="order_id", right_index=True, how="left", validate="one_to_one")
    base = base.merge(
        primary[["primary_seller_id", "primary_category"]],
        left_on="order_id", right_index=True, how="left", validate="one_to_one")
    base = base.merge(pay_agg, left_on="order_id", right_index=True, how="left", validate="one_to_one")
    base = base.merge(rev_agg, left_on="order_id", right_index=True, how="left", validate="one_to_one")
    base = base.merge(geo, left_on="order_id", right_index=True, how="left", validate="one_to_one")

    # Defaults for orders with no items / payments / reviews.
    for col, default in [("n_items", 0), ("has_items", 0), ("n_products", 0),
                         ("n_sellers", 0), ("n_categories", 0),
                         ("seller_id_missing", 0), ("freight_ratio_missing", 1)]:
        base[col] = base[col].fillna(default)
    base["primary_category"] = base["primary_category"].fillna(UNKNOWN)
    base["primary_payment_type"] = base["primary_payment_type"].fillna(UNKNOWN)
    base["n_payment_methods"] = base["n_payment_methods"].fillna(0).astype(int)
    base["payment_missing"] = base["payment_missing"].fillna(1).astype(int)
    base["review_count"] = base["review_count"].fillna(0).astype(int)

    # seller_id_missing override: complete seller set unknown.
    missing_seller = base["seller_id_missing"] == 1
    base.loc[missing_seller, ["distance_km_max", "distance_missing_fraction", "interstate_share"]] = np.nan

    # States (normalized) + primary seller state.
    base["customer_state"] = base["customer_state"].map(normalize_state)
    seller_state_map = sellers.set_index("seller_id")["seller_state"].map(normalize_state)
    base["primary_seller_state"] = base["primary_seller_id"].map(seller_state_map).fillna(UNKNOWN)

    # Time-component features (categorical codes).
    pt = base["order_purchase_timestamp"]
    base["purchase_month"] = pt.dt.month
    base["purchase_dayofweek"] = pt.dt.dayofweek  # Monday=0
    base["purchase_hour"] = pt.dt.hour

    # Targets.
    delivered = base["order_delivered_customer_date"]
    elapsed = (delivered - pt).dt.total_seconds() / 86400.0
    lead_ok = (base["order_status"] == "delivered") & pt.notna() & delivered.notna() & (elapsed > 0)
    base["lead_days"] = np.where(lead_ok, elapsed, np.nan)
    base["regression_label_available_at"] = base["order_delivered_customer_date"].where(lead_ok)

    # Eligibility + exclusion reasons.
    group_resolved = base["customer_unique_id"].notna()
    pred_ok = pt.notna()

    reg_reasons = _join_reasons([
        ("not_delivered", (base["order_status"] != "delivered").to_numpy()),
        ("missing_delivery_timestamp",
         ((base["order_status"] == "delivered") & delivered.isna()).to_numpy()),
        ("missing_prediction_timestamp", (~pred_ok).to_numpy()),
        ("nonpositive_lead_days",
         ((base["order_status"] == "delivered") & pt.notna() & delivered.notna()
          & (elapsed <= 0)).to_numpy()),
        ("missing_customer_group", (~group_resolved).to_numpy()),
    ])
    cls_reasons = _join_reasons([
        ("missing_review_target", base["is_detractor"].isna().to_numpy()),
        ("missing_prediction_timestamp", (~pred_ok).to_numpy()),
        ("missing_customer_group", (~group_resolved).to_numpy()),
    ])

    base["regression_exclusion_reason"] = reg_reasons
    base["classification_exclusion_reason"] = cls_reasons
    base["eligible_regression"] = (base["regression_exclusion_reason"] == "").astype(int)
    base["eligible_classification"] = (base["classification_exclusion_reason"] == "").astype(int)

    # Rename prediction anchor and keep only the contracted column order.
    base = base.rename(columns={"order_purchase_timestamp": "prediction_timestamp"})
    base = base[BASE_COLUMN_ORDER]

    # Nullable integer targets back to Int64 (clean serialization).
    base["review_score_min"] = _to_int64(base["review_score_min"])
    base["is_detractor"] = _to_int64(base["is_detractor"])

    base = base.sort_values("order_id", kind="mergesort").reset_index(drop=True)
    return {
        "base": base,
        "centroids": centroids,
        "geo_filter_counts": geo_filter_counts,
        "polygon": polygon,
        "source": {
            "orders": orders, "items": items, "payments": payments, "reviews": reviews,
            "customers": customers, "products": products, "sellers": sellers,
            "geolocation": geolocation, "translation": translation,
        },
        "order_seller_pairs": order_seller_pairs,
    }


def _to_int64(s: pd.Series) -> pd.Series:
    out = pd.Series(pd.NA, index=s.index, dtype="Int64")
    out[s.notna()] = pd.to_numeric(s[s.notna()], errors="coerce").astype("Int64")
    return out


# ---------------------------------------------------------------------------
# Serialization
# ---------------------------------------------------------------------------

_DATE_COLS = ["prediction_timestamp", "regression_label_available_at",
              "review_latest_observed_at"]


def _serialize_date_col(df: pd.DataFrame, col: str) -> None:
    df[col] = [t.strftime("%Y-%m-%d %H:%M:%S") if pd.notna(t) else "" for t in df[col]]


def export(repo_root: Path, built: dict) -> None:
    ml = repo_root / "data" / "business" / "ml"
    base = built["base"]

    # Fixed column order + date serialization, then stable CSV.
    out = base.copy()
    for col in _DATE_COLS:
        _serialize_date_col(out, col)
    csv_path = ml / "orders_ml_features.csv"
    out.to_csv(csv_path, index=False)

    (ml / "feature_schema.json").write_text(
        json.dumps(_enriched_schema(), indent=2) + "\n", encoding="utf-8")

    # Input hashes (content identity) + boundary provenance.
    pre = repo_root / "data" / "preprocessed"
    input_names = [
        "olist_orders_dataset.csv", "olist_order_items_dataset.csv",
        "olist_order_payments_dataset.csv", "olist_order_reviews_dataset.csv",
        "olist_customers_dataset.csv", "olist_products_dataset.csv",
        "olist_sellers_dataset.csv", "olist_geolocation_dataset.csv",
        "product_category_name_translation.csv",
    ]
    input_hashes = {n: sha256_file(pre / n) for n in input_names}

    geo_dir = repo_root / "docs" / "references" / "geography"
    boundary_manifest = json.loads((geo_dir / "boundary_manifest.json").read_text(encoding="utf-8"))

    manifest = {
        "feature_version": FEATURE_VERSION,
        "contract_version": CONTRACT_VERSION,
        "split_version": SPLIT_VERSION,
        "config": {
            "earth_radius_km": 6371.0,
            "usable_payment_types": sorted(["credit_card", "boleto", "voucher", "debit_card"]),
            "boundary_product": boundary_manifest["product"],
            "boundary_archive_sha256": boundary_manifest["archive_sha256"],
            "boundary_derived_geometry_sha256": boundary_manifest["derived_geometry_sha256"],
            "source_crs": boundary_manifest["original_crs"],
            "target_crs": boundary_manifest["target_crs"],
            "library_versions": boundary_manifest["library_versions"],
        },
        "input_hashes": input_hashes,
        "row_counts": {
            "source_orders": int(len(built["source"]["orders"])),
            "output_orders": int(len(base)),
            "orders_with_items": int((base["has_items"] == 1).sum()),
            "eligible_regression": int(base["eligible_regression"].sum()),
            "eligible_classification": int(base["eligible_classification"].sum()),
        },
        "content_checksum": sha256_file(csv_path),
        "run_timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    (ml / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")


# ---------------------------------------------------------------------------
# Data-level checks + quality report
# ---------------------------------------------------------------------------

def _data_checks(built: dict) -> list[dict]:
    base = built["base"]
    src = built["source"]
    results = []

    def add(fid, ok, detail):
        results.append({"id": fid, "status": "pass" if ok else "fail", "detail": detail})

    add("one_row_per_order", len(base) == len(src["orders"]),
        f"{len(base)} output rows vs {len(src['orders'])} source orders")
    add("unique_nonnull_order_id", base["order_id"].notna().all() and base["order_id"].is_unique,
        "order_id unique and non-null")
    add("join_cardinality_items",
        (base["has_items"] == 1).sum() == src["items"]["order_id"].nunique(),
        "orders with items match distinct item order_ids")
    add("no_cartesian_multiplier", True,
        "items/payments/reviews aggregated to order grain before joins (no multiplication)")
    add("has_items_coherent",
        ((base["has_items"] == 1) == (base["n_items"] > 0)).all(),
        "has_items == int(n_items > 0) for every row")
    add("missing_review_stays_null",
        ((base["review_count"] == 0) == base["is_detractor"].isna()).all(),
        "no review -> null detractor; any review -> non-null detractor")
    add("delivered_unreviewed_is_regression_candidate",
        bool(((base["order_status"] == "delivered") & base["review_count"].eq(0)
              & base["eligible_regression"].eq(1)).any()),
        "some delivered unreviewed orders are regression-eligible")
    add("reviewed_nondelivered_is_classification_candidate",
        bool(((base["order_status"] != "delivered") & base["review_count"].gt(0)
              & base["eligible_classification"].eq(1)).any()),
        "some reviewed non-delivered orders are classification-eligible")
    add("freight_ratio_missing_coherent",
        (base["freight_ratio_missing"] == base["freight_ratio"].isna().astype(int)).all(),
        "freight_ratio_missing == int(freight_ratio is null)")
    add("payment_missing_coherent",
        (base["payment_missing"] == (base["primary_payment_type"] == UNKNOWN).astype(int)).all(),
        "payment_missing == int(primary_payment_type is Unknown)")

    return results


def _missingness(base: pd.DataFrame) -> dict:
    return {c: int(base[c].isna().sum()) for c in PREDICTOR_ALLOWLIST}


def _cohort_counts(base: pd.DataFrame) -> dict:
    n = len(base)
    return {
        "orders": int(n),
        "delivered": int((base["order_status"] == "delivered").sum()),
        "with_items": int((base["has_items"] == 1).sum()),
        "multi_item": int((base["n_items"] > 1).sum()),
        "multi_seller": int((base["n_sellers"] > 1).sum()),
        "with_review": int((base["review_count"] > 0).sum()),
        "detractor": int((base["is_detractor"] == 1).sum()),
        "non_detractor": int((base["is_detractor"] == 0).sum()),
        "eligible_regression": int(base["eligible_regression"].sum()),
        "eligible_classification": int(base["eligible_classification"].sum()),
        "regression_exclusions": base["regression_exclusion_reason"].value_counts().to_dict(),
        "classification_exclusions": base["classification_exclusion_reason"].value_counts().to_dict(),
    }


def _write_quality_report(repo_root: Path, built: dict, fixture_results: list[dict]) -> None:
    ml = repo_root / "data" / "business" / "ml"
    base = built["base"]
    geo_filter_counts = built["geo_filter_counts"]
    centroids = built["centroids"]

    data_checks = _data_checks(built)
    all_checks = fixture_results + data_checks

    # Coverage: customer/seller keys resolved before (>=1 surviving coord) vs after.
    resolved_centroids = centroids[centroids["resolution_status"] == "resolved"]

    report = {
        "checks": all_checks,
        "checks_summary": {
            "passed": sum(1 for c in all_checks if c["status"] == "pass"),
            "failed": sum(1 for c in all_checks if c["status"] == "fail"),
        },
        "geography": {
            "filter_counts": {str(k): int(v) for k, v in geo_filter_counts.items()},
            "zero_survivor_keys": int((centroids["resolution_status"] == "no_surviving_coordinates").sum()),
            "out_of_country_median_keys": int((centroids["resolution_status"] == "median_outside_polygon").sum()),
            "resolved_keys": int(len(resolved_centroids)),
        },
        "cohort_counts": _cohort_counts(base),
        "predictor_missingness": _missingness(base),
        "deferred_checks": [
            "pipeline_fit_fold_only", "history_fixture", "temporal_fixture",
            "cascade_fixture", "reload_pipeline_threshold",
        ],
    }
    (ml / "quality_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")

    failed = [c for c in all_checks if c["status"] == "fail"]
    if failed:
        raise RuntimeError(
            "Required checks failed; publication stopped.\n"
            + "\n".join(f"- {c['id']}: {c['detail']}" for c in failed))


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def _repo_root() -> Path:
    return Path(__file__).resolve().parents[2]


def main() -> None:
    repo = _repo_root()
    built = build(repo)

    # Fixture checks run against the prepared boundary + real export columns.
    fixture_results = run_fixture_checks(built["polygon"], BASE_COLUMN_ORDER)

    export(repo, built)
    _write_quality_report(repo, built, fixture_results)

    ml = repo / "data" / "business" / "ml"
    base = built["base"]
    print("build_features complete.")
    print(f"  orders:      {len(base)} rows -> {ml / 'orders_ml_features.csv'}")
    print(f"  centroids:   {len(built['centroids'])} keys -> {ml / 'zip_centroids.csv'}")
    print(f"  eligible_regression:    {int(base['eligible_regression'].sum())}")
    print(f"  eligible_classification: {int(base['eligible_classification'].sum())}")
    print(f"  fixture+data checks: {sum(1 for c in fixture_results if c['status']=='pass')}/"
          f"{len(fixture_results)} fixtures passed (see quality_report.json)")


if __name__ == "__main__":
    main()

```


### `src/features/create_splits.py`

Split-manifest builder: outer StratifiedGroupKFold development/holdout and inner per-task CV folds.

```python
"""Split-manifest builder (feature_contract.md v1.1 §7, Phase B).

Entrypoint: ``python -m src.features.create_splits``

Reads the base table produced by ``build_features`` and writes, under
``data/business/ml/``:

    split_assignments.csv   one row per order: group, version, development/holdout/excluded
    cv_assignments.csv      one row per task and eligible development order: validation fold

Outer assignment: StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
on resolved customer groups with three split-only strata (negative / nonnegative /
unknown review); fold 0 is the approximate 20% holdout, the rest development.
Inner task folds are independent five-fold splits within development
(StratifiedGroupKFold for classification, GroupKFold for regression).
"""
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold

from .contract import SPLIT_VERSION

OUTER_SPLITS = 5
OUTER_SEED = 42
INNER_SPLITS = 5


def _repo_root() -> Path:
    return Path(__file__).resolve().parents[2]


def _group_class(neg: bool, nonneg: bool) -> str:
    """Split-only stratum: any negative review dominates; else nonnegative; else unknown."""
    if neg:
        return "negative"
    if nonneg:
        return "nonnegative"
    return "unknown"


def build_splits(ml: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    base = pd.read_csv(
        ml / "orders_ml_features.csv",
        usecols=["order_id", "customer_unique_id", "eligible_regression",
                 "eligible_classification", "is_detractor"],
    )
    base["_det"] = pd.to_numeric(base["is_detractor"], errors="coerce")

    resolved = base[base["customer_unique_id"].notna()].copy()
    excluded = base[base["customer_unique_id"].isna()].copy()

    # One stratum per group.
    agg = resolved.groupby("customer_unique_id")["_det"].agg(
        neg=lambda s: bool((s == 1).any()),
        nonneg=lambda s: bool((s == 0).any()),
    ).reset_index()
    agg["stratum"] = [ _group_class(r.neg, r.nonneg) for r in agg.itertuples() ]

    groups_sorted = agg.sort_values("customer_unique_id", kind="mergesort").reset_index(drop=True)
    y = groups_sorted["stratum"].to_numpy()
    gids = groups_sorted["customer_unique_id"].to_numpy()

    sgkf = StratifiedGroupKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=OUTER_SEED)
    X = np.zeros((len(gids), 1))
    holdout_groups = set()
    for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=gids)):
        if fold == 0:
            holdout_groups = set(gids[test_idx])

    group_assign = groups_sorted["customer_unique_id"].map(
        lambda g: "holdout" if g in holdout_groups else "development")

    assignments = resolved[["order_id", "customer_unique_id"]].copy()
    assignments["split_assignment"] = assignments["customer_unique_id"].map(
        lambda g: "holdout" if g in holdout_groups else "development")
    assignments["split_version"] = SPLIT_VERSION
    assignments["exclusion_reason"] = ""

    ex = excluded[["order_id", "customer_unique_id"]].copy()
    ex["split_assignment"] = "excluded"
    ex["split_version"] = SPLIT_VERSION
    ex["exclusion_reason"] = "missing_customer_group"

    assignments = pd.concat([assignments, ex], ignore_index=True)
    assignments = assignments.sort_values("order_id", kind="mergesort").reset_index(drop=True)

    # Inner task folds within development eligible rows.
    dev = resolved[resolved["customer_unique_id"].map(
        lambda g: "development" if g not in holdout_groups else "holdout") == "development"].copy()

    cv_rows = []
    # Classification: StratifiedGroupKFold on binary is_detractor.
    cls_dev = dev[dev["eligible_classification"] == 1].copy()
    if len(cls_dev):
        cls_dev = cls_dev.sort_values("customer_unique_id", kind="mergesort").reset_index(drop=True)
        c_y = cls_dev["_det"].astype(int).to_numpy()  # 0/1 only (eligible => non-null)
        c_g = cls_dev["customer_unique_id"].to_numpy()
        cskf = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=OUTER_SEED)
        for fold, (_, test_idx) in enumerate(cskf.split(np.zeros((len(cls_dev), 1)), c_y, groups=c_g)):
            cv_rows.append(pd.DataFrame({
                "order_id": cls_dev["order_id"].iloc[test_idx].to_numpy(),
                "task": "classification",
                "validation_fold": fold,
            }))

    # Regression: GroupKFold.
    reg_dev = dev[dev["eligible_regression"] == 1].copy()
    if len(reg_dev):
        reg_dev = reg_dev.sort_values("customer_unique_id", kind="mergesort").reset_index(drop=True)
        r_g = reg_dev["customer_unique_id"].to_numpy()
        gkf = GroupKFold(n_splits=INNER_SPLITS)
        for fold, (_, test_idx) in enumerate(gkf.split(np.zeros((len(reg_dev), 1)), groups=r_g)):
            cv_rows.append(pd.DataFrame({
                "order_id": reg_dev["order_id"].iloc[test_idx].to_numpy(),
                "task": "regression",
                "validation_fold": fold,
            }))

    cv = pd.concat(cv_rows, ignore_index=True)
    cv["split_version"] = SPLIT_VERSION
    cv = cv.sort_values(["task", "order_id"], kind="mergesort").reset_index(drop=True)

    return assignments, cv


def _check_splits(assignments: pd.DataFrame, cv: pd.DataFrame,
                  base: pd.DataFrame) -> dict:
    results = []

    def add(fid, ok, detail):
        results.append({"id": fid, "status": "pass" if ok else "fail", "detail": detail})

    holdout = set(assignments.loc[assignments["split_assignment"] == "holdout", "customer_unique_id"])
    dev = set(assignments.loc[assignments["split_assignment"] == "development", "customer_unique_id"])
    add("outer_groups_disjoint", holdout.isdisjoint(dev),
        f"holdout {len(holdout)} groups disjoint from development {len(dev)} groups")

    # Both classification classes present in development and in each validation fold.
    cls_dev = cv[cv["task"] == "classification"].merge(
        base[["order_id", "is_detractor"]], on="order_id", how="left")
    if len(cls_dev):
        classes = sorted(pd.to_numeric(cls_dev["is_detractor"], errors="coerce").dropna().unique())
        add("classification_both_classes_development", set(classes) == {0.0, 1.0},
            f"development classification classes {classes}")
        per_fold_ok = all(
            set(pd.to_numeric(g["is_detractor"], errors="coerce").dropna().unique()) == {0.0, 1.0}
            for _, g in cls_dev.groupby("validation_fold"))
        add("classification_both_classes_each_fold", bool(per_fold_ok),
            "both classes present in every classification validation fold")

    # No holdout order appears in any CV assignment.
    holdout_orders = set(assignments.loc[assignments["split_assignment"] == "holdout", "order_id"])
    cv_orders = set(cv["order_id"])
    add("no_holdout_in_cv", holdout_orders.isdisjoint(cv_orders),
        "no holdout order assigned to an inner validation fold")

    return results


def main() -> None:
    repo = _repo_root()
    ml = repo / "data" / "business" / "ml"

    base = pd.read_csv(ml / "orders_ml_features.csv",
                       usecols=["order_id", "customer_unique_id", "is_detractor",
                                "eligible_regression", "eligible_classification"])
    assignments, cv = build_splits(ml)

    assignments.to_csv(ml / "split_assignments.csv", index=False)
    cv.to_csv(ml / "cv_assignments.csv", index=False)

    checks = _check_splits(assignments, cv, base)
    report = {
        "split_version": SPLIT_VERSION,
        "library_versions": {"scikit-learn": _sklearn_version()},
        "outer_config": {"n_splits": OUTER_SPLITS, "shuffle": True, "random_state": OUTER_SEED,
                         "holdout_fold": 0, "strata": ["negative", "nonnegative", "unknown"]},
        "counts": {
            "holdout_orders": int((assignments["split_assignment"] == "holdout").sum()),
            "development_orders": int((assignments["split_assignment"] == "development").sum()),
            "excluded_orders": int((assignments["split_assignment"] == "excluded").sum()),
            "cv_classification_rows": int((cv["task"] == "classification").sum()),
            "cv_regression_rows": int((cv["task"] == "regression").sum()),
        },
        "checks": checks,
        "run_timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    (ml / "split_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")

    failed = [c for c in checks if c["status"] == "fail"]
    if failed:
        raise RuntimeError(
            "Split checks failed; publication stopped.\n"
            + "\n".join(f"- {c['id']}: {c['detail']}" for c in failed))

    print("create_splits complete.")
    print(f"  split_assignments.csv: {len(assignments)} orders "
          f"(holdout {int((assignments['split_assignment']=='holdout').sum())}, "
          f"dev {int((assignments['split_assignment']=='development').sum())}, "
          f"excluded {int((assignments['split_assignment']=='excluded').sum())})")
    print(f"  cv_assignments.csv: {len(cv)} rows")


def _sklearn_version() -> str:
    import sklearn
    return sklearn.__version__


if __name__ == "__main__":
    main()

```


## 3. Phase A — build the base table

`python -m src.features.build_features` (exact CLI path). This loads the
preprocessed inputs, validates primary keys, prepares the IBGE boundary, builds
the polygon-filtered ZIP/state centroid lookup, aggregates items/payments/reviews
to the order grain, computes distance/interstate geography features, assembles
the one-row-per-order base table, exports the CSV/schema/manifest and runs the
§9 acceptance checks. It fails loudly on any required check failure.


In [ ]:
from src.features import build_features as bf

# Exact equivalent of `python -m src.features.build_features` (build_features.main).
bf.main()


In [ ]:
import json
import pandas as pd

base = pd.read_csv(ML / "orders_ml_features.csv", low_memory=False)
print("orders_ml_features.csv:", base.shape)
print("columns:", list(base.columns))
print()
print("base table preview:")
print(base[["order_id", "n_items", "total_price", "distance_km_max",
            "primary_payment_type", "lead_days", "is_detractor",
            "eligible_regression", "eligible_classification"]].head().to_string())

centroids = pd.read_csv(ML / "zip_centroids.csv")
print()
print("zip_centroids.csv:", centroids.shape)
print(centroids.head(3).to_string())

q = json.loads((ML / "quality_report.json").read_text(encoding="utf-8"))
print()
print("checks summary:", q["checks_summary"])
print("geography filter_counts:", q["geography"]["filter_counts"])
print()
print("cohort counts:")
for k, v in q["cohort_counts"].items():
    print(f"  {k}: {v}")
print()
print("predictor missingness (non-zero only):")
for k, v in q["predictor_missingness"].items():
    if v:
        print(f"  {k}: {v}")
print()
print("deferred checks:", q["deferred_checks"])


## 4. Phase B — build the split manifests

`python -m src.features.create_splits` (exact CLI path). Reads the base table,
assigns each order to `development`/`holdout`/`excluded` via outer
`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` on resolved
customer groups with three split-only strata (negative / nonnegative / unknown);
fold 0 is the holdout. Inner task folds are independent five-fold splits within
development (StratifiedGroupKFold for classification, GroupKFold for regression).


In [ ]:
from src.features import create_splits as cs

# Exact equivalent of `python -m src.features.create_splits` (create_splits.main).
cs.main()


In [ ]:
import json
import pandas as pd

sr = json.loads((ML / "split_report.json").read_text(encoding="utf-8"))
print("split_version:", sr["split_version"])
print("counts:", json.dumps(sr["counts"], indent=2))
print("checks:")
for c in sr["checks"]:
    print(f"  [{c['status']:4s}] {c['id']} — {c['detail']}")

sa = pd.read_csv(ML / "split_assignments.csv")
print()
print("split_assignments.csv:", sa.shape)
print(sa["split_assignment"].value_counts().to_string())

cv = pd.read_csv(ML / "cv_assignments.csv")
print()
print("cv_assignments.csv:", cv.shape)
print(cv.groupby("task")["validation_fold"].value_counts().to_string())


## 5. Reproducibility

- The data files (`orders_ml_features.csv`, `zip_centroids.csv`,
  `split_assignments.csv`, `cv_assignments.csv`) are byte-identical across runs
  (no timestamps in the data). The JSON reports carry `run_timestamp` /
  `retrieved_at` metadata fields, so only those fields differ between runs.
- Deterministic seeds: outer/inner CV use `random_state=42`; primary item/payment
  tie-breaks and all sort orders use `kind="mergesort"`.
- To run without the notebook:

  ```
  python -m src.features.build_features
  python -m src.features.create_splits
  ```

- Training-fitted imputation/scaling/encoding, seller outcome histories and
  `T_pred` are intentionally **not** implemented this round; the §9 checks that
  depend on them are recorded under `quality_report.json → deferred_checks`.
